# Reversed Headline Quality Evaluation

This notebook evaluates `reversed_headline` from `Sarcasm_Headlines_Dataset_v2_flipped.json` with the Hugging Face model `helinivan/english-sarcasm-detector`.

Workflow:
- Run inference only on `reversed_headline`.
- Derive the expected label by flipping the original label (`1 - label` or `1 - is_sarcastic`).
- Compare detector predictions against the flipped label to estimate reversal quality.


In [ ]:
%%capture
!pip -q install transformers scikit-learn pandas matplotlib tqdm


The model card for `helinivan/english-sarcasm-detector` states that label `0` means not sarcastic and label `1` means sarcastic. Its example code also lowercases text and strips punctuation before tokenization, so the notebook follows that preprocessing.


In [ ]:
from pathlib import Path
import string

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pd.set_option("display.max_colwidth", 200)

MODEL_NAME = "helinivan/english-sarcasm-detector"
DATASET_FILENAME = "Sarcasm_Headlines_Dataset_v2_flipped.json"
TEXT_COLUMN = "reversed_headline"
FLIPPED_LABEL_COLUMN = "reversed_label"
MAX_LENGTH = 256
BATCH_SIZE = 64
SAMPLE_SIZE = None  # Set to an integer for a quicker Colab debug run.

candidate_paths = [
    Path("/content/CS4248_group_project/data_generation") / DATASET_FILENAME,
    Path("/content/data_generation") / DATASET_FILENAME,
    Path.cwd() / "data_generation" / DATASET_FILENAME,
    Path.cwd() / DATASET_FILENAME,
]

DATASET_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Upload the file or place the repo under /content/CS4248_group_project, then rerun this cell."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dataset path: {DATASET_PATH}")
print(f"Using device: {device}")


In [ ]:
try:
    df = pd.read_json(DATASET_PATH, lines=True)
except ValueError:
    df = pd.read_json(DATASET_PATH)

if "label" in df.columns:
    original_label_column = "label"
elif "is_sarcastic" in df.columns:
    original_label_column = "is_sarcastic"
else:
    raise KeyError("Expected either 'label' or 'is_sarcastic' in the dataset.")

required_columns = {TEXT_COLUMN, original_label_column}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=[TEXT_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df = df[df[TEXT_COLUMN] != ""].copy()
df[original_label_column] = df[original_label_column].astype(int)

invalid_labels = set(df[original_label_column].unique()) - {0, 1}
if invalid_labels:
    raise ValueError(f"Original label column must contain only 0/1 values. Found: {sorted(invalid_labels)}")

df[FLIPPED_LABEL_COLUMN] = 1 - df[original_label_column]

if SAMPLE_SIZE is not None:
    df = df.head(SAMPLE_SIZE).copy()

preview_columns = [col for col in [original_label_column, FLIPPED_LABEL_COLUMN, "headline", TEXT_COLUMN] if col in df.columns]
display(df[preview_columns].head())
print(f"Rows to evaluate: {len(df):,}")
print(df[FLIPPED_LABEL_COLUMN].value_counts().sort_index().rename(index={0: "not_sarcastic", 1: "sarcastic"}))


In [ ]:
def preprocess_data(text: str) -> str:
    return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

id2label = {0: "not_sarcastic", 1: "sarcastic"}
print(f"Loaded model: {MODEL_NAME}")
print(f"Model config labels: {getattr(model.config, 'id2label', {})}")

def predict_sarcasm(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    predictions = []
    confidences = []
    sarcastic_probabilities = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch_texts = [preprocess_data(text) for text in texts[start:start + batch_size]]
        tokenized = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            logits = model(**tokenized).logits
            probabilities = torch.softmax(logits, dim=-1)

        batch_confidences, batch_predictions = probabilities.max(dim=-1)
        predictions.extend(batch_predictions.cpu().tolist())
        confidences.extend(batch_confidences.cpu().tolist())
        sarcastic_probabilities.extend(probabilities[:, 1].cpu().tolist())

    return predictions, confidences, sarcastic_probabilities


In [ ]:
predictions, confidences, sarcastic_probabilities = predict_sarcasm(df[TEXT_COLUMN].tolist())

df["predicted_label"] = predictions
df["prediction_confidence"] = confidences
df["sarcastic_probability"] = sarcastic_probabilities
df["predicted_label_name"] = df["predicted_label"].map(id2label)
df["expected_label_name"] = df[FLIPPED_LABEL_COLUMN].map(id2label)

display(df[[TEXT_COLUMN, FLIPPED_LABEL_COLUMN, "predicted_label", "prediction_confidence"]].head())


In [ ]:
y_true = df[FLIPPED_LABEL_COLUMN]
y_pred = df["predicted_label"]

summary = pd.DataFrame(
    [
        {
            "rows_evaluated": len(df),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "expected_sarcastic_rate": y_true.mean(),
            "predicted_sarcastic_rate": y_pred.mean(),
        }
    ]
)
display(summary)

report = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        target_names=["not_sarcastic", "sarcastic"],
        output_dict=True,
        zero_division=0,
    )
).transpose()
display(report)


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["not sarcastic", "sarcastic"],
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Reversed headline prediction vs flipped label")
plt.show()


In [ ]:
errors = df[df["predicted_label"] != df[FLIPPED_LABEL_COLUMN]].copy()
errors = errors.sort_values("prediction_confidence", ascending=False)

print(f"Misclassified reversed headlines: {len(errors):,}")
error_columns = [
    col
    for col in [
        "headline",
        TEXT_COLUMN,
        original_label_column,
        FLIPPED_LABEL_COLUMN,
        "predicted_label",
        "prediction_confidence",
        "sarcastic_probability",
    ]
    if col in errors.columns
]
display(errors[error_columns].head(25))

output_path = DATASET_PATH.with_name(DATASET_PATH.stem + "_reversed_headline_eval.csv")
df.to_csv(output_path, index=False)
print(f"Saved full evaluation output to: {output_path}")
